# Fruits & Légumes → PostgreSQL

Pipeline complet : chargement des données fruits & légumes, traitement, puis export dans PostgreSQL (Docker).

## 1. Fix encodage Windows (⚠️ à exécuter EN PREMIER)

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

print(repr(os.getenv("POSTGRES_PASSWORD")))
print(repr(os.getenv("POSTGRES_USER")))
print(repr(os.getenv("POSTGRES_DB")))

'admin123'
'admin'
'db'


In [2]:
# Fix obligatoire sur Windows : psycopg2 plante sur les messages d'erreur
# PostgreSQL en français (encodage cp1252 vs utf-8)
import os
os.environ["PGCLIENTENCODING"] = "UTF8"
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["LC_ALL"] = "en_US.UTF-8"
os.environ["LANG"] = "en_US.UTF-8"

print("✅ Fix encodage appliqué")

✅ Fix encodage appliqué


## 2. Imports & Configuration

In [3]:
import pandas as pd
import warnings
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from urllib.parse import quote_plus

warnings.filterwarnings('ignore')

# Charger les variables d'environnement depuis .env
load_dotenv()

POSTGRES_USER     = os.getenv("POSTGRES_USER", "admin")
POSTGRES_PASSWORD = quote_plus(os.getenv("POSTGRES_PASSWORD", "mdpadmin"))  # encode les caractères spéciaux
POSTGRES_HOST     = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT", "5432")
POSTGRES_DB       = os.getenv("POSTGRES_DB", "db")

DATABASE_URL = f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"

print(f"✅ Connexion configurée vers : {POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}")

✅ Connexion configurée vers : localhost:5432/db


## 3. Connexion PostgreSQL

In [4]:
import psycopg2
try:
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        dbname="db",
        user="admin",
        password="mdpadmin"
    )
    print("✅ Connecté !")
    conn.close()
except Exception as e:
    print(type(e).__name__, ":", e.args)

UnicodeDecodeError : ('utf-8', b'connection to server at "localhost" (::1), port 5432 failed: FATAL:  authentification par mot de passe \xe9chou\xe9e pour l\'utilisateur  \xab admin \xbb\n', 103, 104, 'invalid continuation byte')


In [5]:
engine = create_engine(
    DATABASE_URL,
    connect_args={"client_encoding": "utf8"}  # force UTF-8 côté psycopg2
)

# Test de connexion
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print("✅ Connecté à PostgreSQL :", result.fetchone()[0])

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe9 in position 103: invalid continuation byte

## 4. Chargement des données fruits & légumes

In [ ]:


df = pd.read_csv('../DATA/CLEAN/fruits_legumes_enrichi.csv' , sep=';', encoding='utf-8')

print(f"📥 Données chargées : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.head()

## 5. Exploration rapide des données

In [ ]:
print("📋 Colonnes :", df.columns.tolist())
print(f"\n📐 Dimensions : {df.shape}")
print(f"\n🔍 Valeurs manquantes :")
print(df.isnull().sum())
print(f"\n📊 Types de données :")
print(df.dtypes)

In [ ]:
df.describe()

## 6. Traitement des données

### 6.1 Nettoyage

In [ ]:
df_clean = df.copy()

# Supprimer les doublons
nb_doublons = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
print(f"🗑️ {nb_doublons} doublon(s) supprimé(s)")

# Nettoyer les noms de colonnes (minuscules, sans espaces)
df_clean.columns = (
    df_clean.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_', regex=False)
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)

print(f"✅ Colonnes nettoyées : {df_clean.columns.tolist()}")
print(f"📐 Dimensions après nettoyage : {df_clean.shape}")
df_clean.head()

### 6.2 Transformations spécifiques (à adapter)

In [ ]:
# ⚠️ Adapte cette cellule à la structure réelle de tes données
# Exemples de transformations courantes :

# Convertir une colonne de dates
# df_clean['date'] = pd.to_datetime(df_clean['date'])

# Renommer des colonnes
# df_clean = df_clean.rename(columns={'anciennom': 'nouveau_nom'})

# Filtrer des valeurs
# df_clean = df_clean[df_clean['categorie'].isin(['fruit', 'legume'])]

# Remplir les valeurs manquantes
# df_clean['prix'] = df_clean['prix'].fillna(df_clean['prix'].median())

print(f"✅ Données prêtes : {df_clean.shape}")
df_clean.head()

## 7. Export vers PostgreSQL

### 7.1 Table principale (données brutes)

In [ ]:
df.to_sql(
    name="fruits_legumes_raw",
    con=engine,
    if_exists="replace",   # 'append' pour ajouter sans écraser
    index=False,
    method="multi",
    chunksize=5000
)

# Vérification
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM fruits_legumes_raw")).scalar()
    print(f"✅ Table 'fruits_legumes_raw' : {count} lignes insérées")

### 7.2 Table nettoyée (données traitées)

In [ ]:
df_clean.to_sql(
    name="fruits_legumes_clean",
    con=engine,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=5000
)

# Vérification
with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM fruits_legumes_clean")).scalar()
    print(f"✅ Table 'fruits_legumes_clean' : {count} lignes insérées")

## 8. Relire les données depuis PostgreSQL

In [ ]:
# Relire la table nettoyée
df_from_db = pd.read_sql("SELECT * FROM fruits_legumes_clean", con=engine)
print(f"📖 Relu depuis PostgreSQL : {df_from_db.shape}")
df_from_db.head()

In [ ]:
# Exemple de requête SQL filtrée
# ⚠️ Adapte 'categorie' et 'fruit' au nom réel de ta colonne
df_fruits = pd.read_sql(
    "SELECT * FROM fruits_legumes_clean WHERE categorie = 'fruit' ORDER BY nom",
    con=engine
)
print(f"🍎 Fruits : {df_fruits.shape}")
df_fruits.head(10)

## 9. Lister les tables de la base

In [ ]:
with engine.connect() as conn:
    tables = conn.execute(text(
        "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'"
    )).fetchall()
    print("📋 Tables dans la base :")
    for t in tables:
        print(f"   - {t[0]}")

## 10. Export CSV (optionnel)

In [ ]:
# df_clean.to_csv("../DATA/processed/fruits_legumes_clean.csv", index=False)
# print("💾 CSV sauvegardé !")